# 03b The THINGS-EEG2 paper's encoder, reproduced

**Kaggle:** GPU T4, Internet on. **Inputs:** none (the EEG and the images come from OSF).

The THINGS-EEG2 paper reports that a linearizing encoder on AlexNet predicts a person's EEG at a peak correlation of
0.67 (110 ms), and a model trained on the other nine people 0.45 (130 ms). This notebook rebuilds that encoder from
the paper's released code (github.com/gifale95/eeg_encoding) step by step and checks that those numbers come out.
Where memory forced a change, it is named and measured. Its image features are the basis of every later EEG
analysis.

In [ ]:
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
import json
import shutil
import time

import numpy as np
import pandas as pd
import torch

from cross_subject_encoding.data import things_eeg2 as te
from cross_subject_encoding.data import things_images as ti
from cross_subject_encoding.eval.metrics import pearson_per_output, time_curve
from cross_subject_encoding.eval.paper_correlation import paper_correlation, paper_stats
from cross_subject_encoding.models import paper_linearizing as pl
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.selfcheck import check, headline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PRETRAINED = True  # the paper's ImageNet weights
WORKERS = min(4, os.cpu_count() or 1)
SUBJECTS = C.EEG_SUBJECTS
torch.use_deterministic_algorithms(True, warn_only=True)  # as in the paper's extraction script
T0 = time.time()
minutes = lambda: f"{(time.time() - T0) / 60:.1f} min"
print(f"device {DEVICE} | {len(SUBJECTS)} subjects | {pl.N_FEATURES} AlexNet values per image | seed {pl.PAPER_SEED}")

## Images, in the EEG's order

The image order comes from the image metadata. The
paper's script sorts the image paths instead; the line below says whether
the two orders agree.

In [ ]:
search = Path("/kaggle/input") if IS_KAGGLE else C.PROJECT_ROOT / "data" / "raw" / "things_images"
roots = {s: ti.find_image_root(search, s) for s in ("training", "test")}
if not all(roots.values()):
    roots = ti.download_images(Path("/kaggle/tmp/things_images") if IS_KAGGLE else search)
meta = te.load_image_metadata(C.EEG_RAW if C.EEG_RAW.is_dir() else None)
paths = {s: ti.image_paths(roots[s], meta, s) for s in ("training", "test")}
all_paths = paths["training"] + paths["test"]
train = np.array([True] * len(paths["training"]) + [False] * len(paths["test"]))
same_order = all(p == sorted(p) for p in paths.values())
print({s: len(p) for s, p in paths.items()}, "| metadata order equals the paper's sorted order:", same_order, "|", minutes())

## AlexNet feature maps

The paper's own wrapper around torchvision's AlexNet, the paper's
preprocessing, and the eight layers it keeps. The values are held in
float16 for memory. The first 500 training images are also kept in
float32, so the effect of that rounding on the kernel can be measured
below.

In [ ]:
x, x32 = pl.extract(all_paths, DEVICE, batch=64, workers=WORKERS, pretrained=PRETRAINED, keep_float32_rows=500,
                    progress=lambda i, n: print(f"  {i}/{n}", end="\r"))
assert x.shape == (len(all_paths), pl.N_FEATURES) and np.isfinite(x).all()
print(f"feature maps {x.shape}, {x.nbytes / 1e9:.1f} GB float16 | {minutes()}")

## Standardization and kernel PCA, fitted on training images

`StandardScaler` statistics from the training images. Then the polynomial
kernel sklearn would compute (degree 4, gamma = 1 / number of features,
coef0 = 1), built from blocks of features on the GPU and passed to
sklearn's `KernelPCA` as a precomputed kernel, which centres it and solves
it exactly as `kernel='poly'` would. 1,000 components, as in the paper.

In [ ]:
mean, scale = pl.standardize_stats(x, train)
g = pl.gram(x, mean, scale, train, DEVICE)
k = pl.poly_kernel(g, pl.N_FEATURES)
del g
# float16 storage against float32, on the first 500 training images
z16 = (x[:len(x32)].astype(np.float64) - mean) / scale
z32 = (x32.astype(np.float64) - mean) / scale
k16, k32 = pl.poly_kernel(z16 @ z16.T, pl.N_FEATURES), pl.poly_kernel(z32 @ z32.T, pl.N_FEATURES)
kernel_rel_err = float(np.abs(k16 - k32).max() / np.abs(k32).max())
del z16, z32, k16, k32, x32
print(f"kernel {k.shape}; largest float16 error relative to the kernel's range: {kernel_rel_err:.2e} | {minutes()}")
n_comp = min(pl.N_COMPONENTS, int(train.sum()) - 1)
z_train, z_test, kpca = pl.kernel_pca(k[train], k[~train], n_comp)
del k, x
np.savez(C.DATA_PROCESSED / "paper_alexnet_kpca1000.npz", train=z_train.astype(np.float32), test=z_test.astype(np.float32),
         image=np.array([f"{p.parent.name}/{p.name}" for p in all_paths]))
print(f"kernel PCA: {z_train.shape[1]} components | {minutes()}")

## EEG, and the paper's regression

Training EEG averaged over its 4 repetitions, as the paper's
`load_eeg_data` does. The test EEG keeps all 80 repetitions, because the
paper's correlation draws a new 40/40 split 100 times. Within a person the
regression is fitted on their own training EEG; across people on the
average of the other nine people's.

In [ ]:
ATTACHED = C.EEG_RAW.is_dir() and any(C.EEG_RAW.rglob("preprocessed_eeg_test.npy"))
WORK = C.EEG_RAW if ATTACHED else Path("/kaggle/tmp/things_eeg2")
y_train, bio_test = {}, {}
for s in SUBJECTS:
    if not ATTACHED and te.subject_file(WORK, s, "test") is None:
        te.download_subject(s, WORK)
    tr = te.load_part(te.subject_file(WORK, s, "training"))
    y_train[s] = tr["data"].mean(1).reshape(tr["data"].shape[0], -1)
    bio_test[s] = te.load_part(te.subject_file(WORK, s, "test"))["data"]
    times = tr["times"]
    del tr
    if not ATTACHED:
        shutil.rmtree(WORK / s, ignore_errors=True)
    print(f"{s}: training {y_train[s].shape}, test {bio_test[s].shape} | {minutes()}")
n_ch, n_t = bio_test[SUBJECTS[0]].shape[2:]
synt = {}
for s in SUBJECTS:
    within = pl.ols_paper(z_train, y_train[s], z_test, DEVICE)
    between = pl.ols_paper(z_train, np.mean([y_train[o] for o in SUBJECTS if o != s], 0), z_test, DEVICE)
    synt[s] = {"within": within.reshape(-1, n_ch, n_t), "between": between.reshape(-1, n_ch, n_t)}
print(f"regressions done | {minutes()}")

## The paper's correlation and noise ceiling

For each person, 100 random 40/40 splits of the test repetitions, the
correlation with one half's average at every channel and time point,
averaged over splits. Then the mean over channels and people, with 95%
intervals from 10,000 resamples of people. The same predictions are also
scored with one fixed 40-repetition draw, to show how
much that detail matters.

In [ ]:
res = {s: paper_correlation(synt[s], bio_test[s]) for s in SUBJECTS}
stats = {name: paper_stats(np.stack([res[s]["correlation"][name] for s in SUBJECTS])) for name in ("within", "between")}
nc = {b: np.stack([res[s][f"noise_ceiling_{b}"] for s in SUBJECTS]).mean((0, 1)) for b in ("low", "up")}
fixed = {}
for name in ("within", "between"):
    curves = []
    for i, s in enumerate(SUBJECTS):
        reps = np.random.default_rng([C.SEED, C.EEG_SUBJECTS.index(s)]).permutation(bio_test[s].shape[1])[:C.PAPER_TEST_REPS]
        y40 = bio_test[s][:, reps].mean(1).reshape(len(bio_test[s]), -1)
        curves.append(time_curve(pearson_per_output(y40, synt[s][name].reshape(len(y40), -1)), n_ch))
    fixed[name] = float(np.max(np.mean(curves, 0)))
table = pd.DataFrame([{"model": name, "peak_r": st["peak"], "ci_lower": st["peak_lo"], "ci_upper": st["peak_hi"],
                       "peak_ms": int(round(times[st["peak_index"]] * 1000)), "paper": {"within": 0.67, "between": 0.45}[name],
                       "peak_r_one_fixed_draw": fixed[name]} for name, st in stats.items()])
table.to_csv(C.METRICS_DIR / "03b_paper_reproduction.csv", index=False)
np.savez_compressed(C.METRICS_DIR / "03b_curves.npz", times=times, nc_low=nc["low"], nc_up=nc["up"],
                    **{f"{n}_{k}": stats[n][k] for n in stats for k in ("curve", "ci_lower", "ci_upper")},
                    **{f"{n}_per_person": np.stack([res[s]["correlation"][n] for s in SUBJECTS]) for n in stats})
print(f"noise ceiling at the within peak: lower {nc['low'][stats['within']['peak_index']]:.3f}, upper {nc['up'][stats['within']['peak_index']]:.3f} | {minutes()}")
table.round(4)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
ms = times * 1000
ax.fill_between(ms, nc["low"], nc["up"], color="0.85", label="noise ceiling (lower to upper)")
for name, colour in (("within", "C0"), ("between", "C1")):
    st = stats[name]
    ax.plot(ms, st["curve"], color=colour, label=f"{name} a person, peak {st['peak']:.2f} at {ms[st['peak_index']]:.0f} ms")
    ax.fill_between(ms, st["ci_lower"], st["ci_upper"], color=colour, alpha=0.25)
for v in (0.67, 0.45):
    ax.axhline(v, color="0.5", ls=":", lw=1)
ax.axvline(0, color="0.6", lw=1)
ax.set(xlabel="ms after image onset", ylabel="r, mean over channels and people",
       title="The paper's AlexNet linearizing encoder, reproduced")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(C.FIGURES_DIR / "03b_paper_reproduction.png", dpi=150)

## Self-check

The published numbers are the expected results. LOOK AGAIN means the
reproduction differs from the paper, and the line under it says where to
look first.

In [ ]:
w, b = stats["within"], stats["between"]
rows = [
    {"measure": "within a person, peak r", "measured": w["peak"], "expected": "0.67 (AlexNet)", "ok": (0.64, 0.70),
     "look_again": "off by more than 0.03: compare each step with the paper's code before anything else"},
    {"measure": "within a person, peak time (ms)", "measured": int(round(times[w["peak_index"]] * 1000)), "expected": "110",
     "ok": (100, 120), "look_again": "elsewhere: the time axis or the image order"},
    {"measure": "across people (trained on the other nine), peak r", "measured": b["peak"], "expected": "0.45",
     "ok": (0.42, 0.48), "look_again": "off by more than 0.03: the averaging of the nine people's EEG"},
    {"measure": "across people, peak time (ms)", "measured": int(round(times[b["peak_index"]] * 1000)), "expected": "130",
     "ok": (120, 140), "look_again": "elsewhere: as above"},
    {"measure": "within a person, mean r before onset", "measured": float(w["curve"][times < 0].mean()), "expected": "about 0",
     "ok": (-0.05, 0.05), "look_again": "clearly above 0: leakage"},
    {"measure": "float16 storage: largest kernel error relative to its range", "measured": kernel_rel_err,
     "expected": "under 1e-3", "ok": (0, 1e-3), "look_again": "keep the feature maps in float32 on disk instead"},
    {"measure": "metadata image order equals the paper's sorted order", "measured": str(same_order), "expected": "True",
     "ok": lambda v: v == "True", "look_again": "the paper's features would then be in sorted order: check which order its EEG uses"},
    {"measure": "one fixed 40-repetition draw minus the paper's 100 splits, within peak", "measured": fixed["within"] - w["peak"],
     "expected": "close to 0", "ok": None, "look_again": ""},
]
status = check("03b", rows, C.METRICS_DIR)
headline("03b", {"within peak r": w["peak"], "within lo": w["peak_lo"], "within hi": w["peak_hi"],
                 "within peak ms": int(round(times[w["peak_index"]] * 1000)),
                 "between peak r": b["peak"], "between lo": b["peak_lo"], "between hi": b["peak_hi"],
                 "between peak ms": int(round(times[b["peak_index"]] * 1000)),
                 "noise ceiling low at within peak": float(nc["low"][w["peak_index"]]),
                 "kernel float16 error": kernel_rel_err}, C.METRICS_DIR)

## Save outputs as a Kaggle Dataset

Publishes every file this notebook wrote to `data/processed/`, `results/metrics/` and `results/figures/` as a new version of the private Kaggle Dataset `cross-subject-brain-encoding-03b-outputs` (created on the first run). The cell first checks that the required outputs exist, so an unfinished run uploads nothing.

Needs Internet on and the Kaggle Secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` attached to this notebook (Add-ons -> Secrets). Without them the cell prints how to set them up and skips.

In [ ]:
from cross_subject_encoding.utils.kaggle_export import export_to_dataset

EXPORT_TO_DATASET = True
if EXPORT_TO_DATASET:
    export_to_dataset(
        slug="cross-subject-brain-encoding-03b-outputs",
        title="cross-subject-brain-encoding 03b outputs",
        files=[C.DATA_PROCESSED / "paper_alexnet_kpca1000.npz", C.METRICS_DIR / "03b_paper_reproduction.csv", C.METRICS_DIR / "03b_curves.npz", C.FIGURES_DIR / "03b_paper_reproduction.png", C.METRICS_DIR / "03b_selfcheck.csv", C.METRICS_DIR / "03b_headline.json"],
        include_dirs=[C.DATA_PROCESSED, C.METRICS_DIR, C.FIGURES_DIR],  # everything else this notebook wrote
        message="notebook run",
    )